# Tutorial B01: S1-GRiTS Full-Year Monthly Mosaic - 3x4 Panel Figure

**Purpose:** Produce a 3-row x 4-column publication figure showing all 12 months of a
selected year from an S1-GRiTS output dataset.

---

## Workflow

1. **User configuration** - output directory, year, display options
2. **VRT generation** - one `s1grits mosaic create` call per month (skips existing VRTs)
3. **Global radiometric baseline** - percentile stretch computed from a reference month
   (or loaded from a saved JSON), applied uniformly to all 12 panels
4. **Phase 1** - generate per-month raw RGBA PNGs (Dask I/O, cached to disk)
5. **Phase 2** - assemble and save the final 3x4 publication figure

---

## Layout

```
| Jan. | Feb. | Mar. | Apr. |
| May. | Jun. | Jul. | Aug. |
| Sep. | Oct. | Nov. | Dec. |
```

---

## Notes

- Months with no processed data are shown as grey placeholder panels.
- The stretch baseline is **fixed across all 12 panels** so seasonal comparisons
  are radiometrically consistent.
- Re-runs skip already-cached VRTs and panel PNGs for speed.


## 1. Imports


In [ ]:
import json
import re
import shutil
import subprocess
import time
import warnings
from pathlib import Path

import numpy as np
import rasterio
import rioxarray
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from osgeo import gdal
from PIL import Image
import geopandas as gpd

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Arial", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

print("INFO - Imports OK")


## 2. Configuration

Edit the variables in this cell to match your dataset.
All subsequent cells use only the constants defined here.


In [ ]:
# A4 landscape physical dimensions (width x height, inches)
A4_WIDTH_INCHES  = 11.69
A4_HEIGHT_INCHES = 5.5
FIGSIZE = (A4_WIDTH_INCHES, A4_HEIGHT_INCHES)

# Base font size
BASE_FONT = 12

# Apply global Matplotlib font settings
plt.rcParams.update({
    "font.size": BASE_FONT,
    "axes.titlesize": BASE_FONT,
    "axes.labelsize": BASE_FONT,
    "xtick.labelsize": BASE_FONT,
    "ytick.labelsize": BASE_FONT,
    "figure.titlesize": BASE_FONT,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"]
})


In [ ]:
# =============================================================================
# USER INPUT: Dataset paths
# =============================================================================

# Path to the S1-GRiTS output folder (contains per-tile Zarr stores + catalog.parquet)
OUTPUT_DIR   = r"YOUR_OUTPUT_DIR"        # path to s1grits output folder

# Orbit direction used during processing
DIRECTION    = "ASCENDING"               # "ASCENDING" or "DESCENDING"

# Directory where VRT mosaics will be written
MOSAIC_DIR   = r"YOUR_MOSAIC_DIR"        # where VRT mosaics will be written

# Directory where the final figure will be saved
FIGURE_DIR   = r"YOUR_FIGURE_DIR"        # where the final figure will be saved

# Optional boundary shapefile (EPSG:4326). Set to None to skip clipping.
BOUNDARY_SHP = None                      # optional: path to boundary shapefile, or None

# Short study-area tag used in filenames (no spaces)
Study_Area   = "MyStudyArea"             # short tag used in filenames (no spaces)

# =============================================================================
# USER INPUT: Year
# =============================================================================

# Ask the user for the target year (all 12 months of that year will be shown)
_year_input = input("Enter year (e.g. 2025, default 2025): ").strip() or "2025"
if not re.fullmatch(r"\d{4}", _year_input):
    raise ValueError(f"Invalid year '{_year_input}'. Must be a 4-digit number.")
TARGET_YEAR = int(_year_input)

# =============================================================================
# Stretch baseline (global radiometric reference)
# =============================================================================

# Path to a previously saved JSON baseline.
# Set to None to compute fresh percentiles from REF_MONTH.
VTR_BASE = None                          # path to saved JSON baseline, or None to compute fresh

# Reference month used when computing a NEW baseline (zero-padded, 01-12).
# Only used if VTR_BASE is None or the file does not exist.
REF_MONTH = "01"

# =============================================================================
# RGB band assignment
# Band order in 12-band COG/Zarr:
#   0=VV_dB  1=VH_dB  2=Ratio(VH/VV)  3=RVI
#   4=VV_glcm_CONTR  5=VV_glcm_IDM  6=VV_glcm_ENT  7=VV_glcm_CORR
#   8=VH_glcm_CONTR  9=VH_glcm_IDM 10=VH_glcm_ENT 11=VH_glcm_CORR
# =============================================================================
RGB_BANDS  = (0, 1, 4)                         # R=VV_dB, G=VH_dB, B=VV_glcm_CONTR
BAND_NAMES = ("VV_dB", "VH_dB", "VV_glcm_CONTR")

# Use (0, 1, 2) if GLCM bands are not available:
# RGB_BANDS  = (0, 1, 2)   # R=VV_dB, G=VH_dB, B=Ratio
# BAND_NAMES = ("VV_dB", "VH_dB", "Ratio")

# =============================================================================
# Stretch and display parameters
# =============================================================================
P_MIN      = 5       # Lower percentile for stretch
P_MAX      = 95      # Upper percentile for stretch
GAMMA      = 2.0     # Gamma correction (>1 brightens mid-tones)
DOWNSAMPLE = 3       # Spatial downsampling factor for VRT loading (higher = faster)

# =============================================================================
# Figure layout
# =============================================================================
TARGET_CRS  = "EPSG:4326"
N_ROWS      = 3
N_COLS      = 4
FIGSIZE     = (28, 18)   # Width x Height in inches
DPI         = 300        # Final output DPI (use 72 for quick draft)
BASE_FONT   = 16

# =============================================================================
# Month metadata
# =============================================================================
MONTH_ABBR: dict[str, str] = {
    "01": "Jan.", "02": "Feb.", "03": "Mar.", "04": "Apr.",
    "05": "May.", "06": "Jun.", "07": "Jul.", "08": "Aug.",
    "09": "Sep.", "10": "Oct.", "11": "Nov.", "12": "Dec.",
}
ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]   # ["01", "02", ..., "12"]

# Create output directories
Path(MOSAIC_DIR).mkdir(parents=True, exist_ok=True)
Path(FIGURE_DIR).mkdir(parents=True, exist_ok=True)

print(f"INFO - OUTPUT_DIR  : {OUTPUT_DIR}")
print(f"INFO - MOSAIC_DIR  : {MOSAIC_DIR}")
print(f"INFO - FIGURE_DIR  : {FIGURE_DIR}")
print(f"INFO - Target year : {TARGET_YEAR}")
print(f"INFO - Months      : Jan. - Dec. (all 12)")
print(f"INFO - RGB bands   : {RGB_BANDS} -> {BAND_NAMES}")
print(f"INFO - Stretch     : p=[{P_MIN},{P_MAX}]  gamma={GAMMA}")
print(f"INFO - Layout      : {N_ROWS} rows x {N_COLS} cols (3x4)")


## 3. Generate VRTs via CLI

Runs `s1grits mosaic create` for every month of `TARGET_YEAR` and captures the VRT paths.
Already-existing VRTs are **skipped** to avoid redundant reprojection work.


In [ ]:
_s1grits = shutil.which("s1grits") or "s1grits"
VRT_PATHS: dict[str, str] = {}   # month_str -> vrt_path

for month in ALL_MONTHS:
    month_str = f"{TARGET_YEAR}-{month}"

    # Fast path: look for an existing VRT
    existing = sorted(Path(MOSAIC_DIR).glob(f"*-{TARGET_YEAR}-{month}-{DIRECTION}-*.vrt"))
    if existing:
        VRT_PATHS[month_str] = str(existing[0])
        print(f"INFO  [{month_str}] Found existing VRT: {existing[0].name}")
        continue

    # Run CLI to create VRT
    print(f"INFO  [{month_str}] Running CLI...")
    _cmd = [
        _s1grits, "mosaic",
        "--month", month_str,
        "--direction", DIRECTION,
        "--output-dir", str(OUTPUT_DIR),
        "--output", str(MOSAIC_DIR),
        "--crs", TARGET_CRS,
    ]
    try:
        result = subprocess.run(_cmd, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as e:
        print(f"\nERROR - s1grits mosaic failed for {month_str}")
        print(e.stderr if e.stderr and e.stderr.strip() else e.stdout)
        print(f"WARN  [{month_str}] Skipping this month.")
        continue

    time.sleep(0.5)   # Allow file system to flush

    # Locate newly created VRT
    new_vrt = sorted(Path(MOSAIC_DIR).glob(f"*-{TARGET_YEAR}-{month}-{DIRECTION}-*.vrt"))
    if new_vrt:
        VRT_PATHS[month_str] = str(new_vrt[0])
        print(f"INFO  [{month_str}] VRT created: {new_vrt[0].name}")
    else:
        print(f"WARN  [{month_str}] CLI succeeded but VRT not found in {MOSAIC_DIR}")

print(f"\nINFO - Ready VRTs: {len(VRT_PATHS)}/12")
for ms, vp in sorted(VRT_PATHS.items()):
    print(f"       {ms}: {Path(vp).name}")


## 4. Global Radiometric Baseline

All 12 panels share the **same (vmin, vmax)** per band so seasonal brightness
differences are preserved and directly comparable.

Two workflows are supported:

- **Load from JSON** (Cell 4a) - use a previously saved baseline.
  Recommended for multi-region consistency.
- **Compute from reference month** (Cell 4b) - derive fresh percentiles from
  `REF_MONTH` of `TARGET_YEAR`. Run Cell 4b first and then load via Cell 4a afterwards.


In [ ]:
# =============================================================================
# Cell 4a: Load global radiometric baseline from VTR_BASE JSON
#
# ALL 12 panels share the stretch parameters derived from this single reference.
# If VTR_BASE is None, run Cell 4b first to compute and save a baseline.
# =============================================================================

if VTR_BASE is None:
    raise RuntimeError(
        "VTR_BASE is None. "
        "Set VTR_BASE to the JSON baseline path in the Configuration cell, "
        "or run Cell 4b to compute a fresh baseline."
    )

_base_path = Path(VTR_BASE)
if not _base_path.exists():
    raise FileNotFoundError(
        f"Baseline JSON not found: {_base_path}\n"
        f"  Run Cell 4b once to compute and save it, then re-run this cell."
    )

with open(_base_path, "r", encoding="utf-8") as _f:
    baseline_config = json.load(_f)

GLOBAL_STRETCH = baseline_config["stretch_params"]
_meta = baseline_config.get("metadata", {})

print(f"INFO - Baseline loaded from : {_base_path.name}")
print(f"       Reference year       : {_meta.get('reference_year', 'N/A')}")
print(f"       Reference month      : {_meta.get('reference_month', 'N/A')}")
print(f"       Percentiles          : p=[{_meta.get('p_min','?')}, {_meta.get('p_max','?')}]")
print("       Per-band stretch     :")
for key, bname in zip(("r", "g", "b"), BAND_NAMES):
    lo, hi = GLOBAL_STRETCH[key]
    print(f"         {key.upper()} ({bname:>18s}): [{lo:+.3f}, {hi:+.3f}]")


In [ ]:
# =============================================================================
# Cell 4b: Compute baseline from REF_MONTH VRT and optionally save to JSON
#
# Run this cell only if:
#   (a) no saved JSON exists yet, OR
#   (b) you want to recompute from the current dataset
# After running, re-run Cell 4a to load the saved values.
# =============================================================================

def compute_vrt_percentiles(
    vrt_path: str,
    p_min: float,
    p_max: float,
    rgb_bands: tuple,
    scale: float = 0.10,
) -> dict:
    """
    Compute per-band percentile stretch from a VRT using GDAL fast-read.

    Reads a spatially downsampled copy (default 10%) to keep memory low while
    producing statistically representative estimates.

    Args:
        vrt_path  : Path to reference VRT (EPSG:4326).
        p_min     : Lower percentile (e.g. 5).
        p_max     : Upper percentile (e.g. 95).
        rgb_bands : Zero-based (R, G, B) band indices.
        scale     : Fraction of full resolution to sample.

    Returns:
        Dict with keys "r", "g", "b", each a (lo, hi) float tuple.
    """
    ds = gdal.Open(str(vrt_path))
    if ds is None:
        raise FileNotFoundError(f"Cannot open VRT: {vrt_path}")

    out_x = max(1, int(ds.RasterXSize * scale))
    out_y = max(1, int(ds.RasterYSize * scale))

    stats = {}
    for key, band_idx in zip(("r", "g", "b"), rgb_bands):
        band = ds.GetRasterBand(band_idx + 1)   # GDAL is 1-based
        raw = band.ReadAsArray(
            resample_alg=gdal.GRIORA_NearestNeighbour,
            buf_xsize=out_x, buf_ysize=out_y,
        )
        if raw is None:
            raise RuntimeError(
                f"GDAL returned None for band {band_idx + 1}. "
                f"Check that the VRT source files still exist."
            )
        data = raw.astype(np.float32)
        nodata = band.GetNoDataValue()
        if nodata is not None:
            data[data == nodata] = np.nan
        valid = data[np.isfinite(data)]
        stats[key] = (
            (float(np.percentile(valid, p_min)), float(np.percentile(valid, p_max)))
            if valid.size > 0 else (0.0, 1.0)
        )
    ds = None
    return stats


# Use REF_MONTH VRT as the reference
_ref_month_str = f"{TARGET_YEAR}-{REF_MONTH}"
_ref_vrt = VRT_PATHS.get(_ref_month_str)

if _ref_vrt is None:
    print(f"WARN - No VRT found for reference month {_ref_month_str}. Run Cell 3 first.")
else:
    print(f"INFO - Computing baseline from: {Path(_ref_vrt).name}")
    print(f"       Bands: {RGB_BANDS}  p=[{P_MIN}, {P_MAX}]")

    GLOBAL_STRETCH = compute_vrt_percentiles(_ref_vrt, P_MIN, P_MAX, RGB_BANDS)

    print(f"INFO - Computed stretch (reference: {_ref_month_str}):")
    for key, bname in zip(("r", "g", "b"), BAND_NAMES):
        lo, hi = GLOBAL_STRETCH[key]
        print(f"       {key.upper()} ({bname:>18s}): [{lo:+.3f}, {hi:+.3f}]")

    # Save to JSON for reuse across sessions
    _save_path = Path(FIGURE_DIR) / f"sar_stretch_baseline_{Study_Area}_{TARGET_YEAR}_{REF_MONTH}.json"
    _payload = {
        "metadata": {
            "reference_vrt": Path(_ref_vrt).name,
            "reference_year": TARGET_YEAR,
            "reference_month": REF_MONTH,
            "p_min": P_MIN,
            "p_max": P_MAX,
            "bands": BAND_NAMES,
        },
        "stretch_params": GLOBAL_STRETCH,
    }
    with open(_save_path, "w", encoding="utf-8") as _f:
        json.dump(_payload, _f, indent=4)
    print(f"INFO - Baseline saved to: {_save_path}")


## 5. Helper Functions


In [ ]:
def _norm(arr: np.ndarray, lo: float, hi: float, gamma: float) -> np.ndarray:
    """Clip -> linear rescale -> gamma correction -> [0, 1]."""
    arr = arr.astype(np.float32, copy=False)
    if hi <= lo:
        return np.zeros_like(arr)
    out = np.clip((arr - lo) / (hi - lo), 0.0, 1.0)
    if gamma != 1.0:
        out = np.power(out, gamma)
    out[~np.isfinite(arr)] = np.nan
    return out


def save_raw_panel_png(
    month_str: str,
    vrt_path: str,
    out_png: Path,
    out_json: Path,
    boundary_gdf,
) -> None:
    """
    Load VRT with Dask, clip (optional), normalise, and save a lossless RGBA PNG.

    No cartographic elements are drawn here; they are added in Phase 2.
    Geographic extent is stored in a JSON sidecar for Phase 2 geo-registration.
    """
    ds = rioxarray.open_rasterio(vrt_path, chunks={"x": 2048, "y": 2048}, masked=True)

    if DOWNSAMPLE > 1:
        ds = ds.coarsen(x=DOWNSAMPLE, y=DOWNSAMPLE, boundary="trim").mean()

    if boundary_gdf is not None:
        try:
            ds = ds.rio.clip(boundary_gdf.geometry, ds.rio.crs, drop=True, all_touched=False)
        except Exception as e:
            print(f"WARN  [{month_str}] Clip failed: {e}")

    xmin = float(ds.x.min())
    xmax = float(ds.x.max())
    ymin = float(ds.y.min())
    ymax = float(ds.y.max())

    idx_r, idx_g, idx_b = RGB_BANDS
    # .compute() forces Dask evaluation immediately for this VRT
    r_data = ds.isel(band=idx_r).compute().values.astype(np.float32)
    g_data = ds.isel(band=idx_g).compute().values.astype(np.float32)
    b_data = ds.isel(band=idx_b).compute().values.astype(np.float32)
    ds.close()

    valid = np.isfinite(r_data) & np.isfinite(g_data) & np.isfinite(b_data)
    alpha = valid.astype(np.float32)

    r_norm = _norm(r_data, GLOBAL_STRETCH["r"][0], GLOBAL_STRETCH["r"][1], GAMMA)
    g_norm = _norm(g_data, GLOBAL_STRETCH["g"][0], GLOBAL_STRETCH["g"][1], GAMMA)
    b_norm = _norm(b_data, GLOBAL_STRETCH["b"][0], GLOBAL_STRETCH["b"][1], GAMMA)

    rgba = np.dstack((
        np.nan_to_num(r_norm, nan=0.0),
        np.nan_to_num(g_norm, nan=0.0),
        np.nan_to_num(b_norm, nan=0.0),
        alpha,
    ))

    # Save lossless RGBA PNG
    rgba_u8 = (rgba * 255).clip(0, 255).astype(np.uint8)
    Image.fromarray(rgba_u8, mode="RGBA").save(str(out_png))

    # Save geographic extent as JSON sidecar
    with open(out_json, "w") as _f:
        json.dump({"extent": [xmin, xmax, ymin, ymax], "month": month_str}, _f)


print("INFO - Helper functions defined")


## 6. Load Boundary (Optional)


In [ ]:
# Load boundary shapefile for spatial clipping (optional)
if BOUNDARY_SHP is not None:
    boundary_gdf = gpd.read_file(BOUNDARY_SHP).to_crs("EPSG:4326")
    print(f"INFO - Boundary loaded: {len(boundary_gdf)} feature(s)")
else:
    boundary_gdf = None
    print("INFO - No boundary shapefile. Full VRT extent will be used.")


## 7. Phase 1: Generate Per-Month RGBA PNGs

Reads each VRT with Dask, applies the global stretch, and saves a lossless RGBA PNG.
Already-cached PNGs are skipped on re-runs.


In [ ]:
_cache_dir = Path(FIGURE_DIR) / f"_panel_cache_{Study_Area}_{TARGET_YEAR}"
_cache_dir.mkdir(parents=True, exist_ok=True)

PANEL_PNGS: dict[str, Path] = {}
PANEL_EXTENTS: dict[str, list] = {}

for month in ALL_MONTHS:
    month_str = f"{TARGET_YEAR}-{month}"
    out_png  = _cache_dir / f"{month_str}.png"
    out_json = _cache_dir / f"{month_str}.json"

    if out_png.exists() and out_json.exists():
        with open(out_json) as _f:
            _meta = json.load(_f)
        PANEL_PNGS[month_str]    = out_png
        PANEL_EXTENTS[month_str] = _meta["extent"]
        print(f"INFO  [{month_str}] Cached PNG found")
        continue

    vrt_path = VRT_PATHS.get(month_str)
    if vrt_path is None:
        print(f"WARN  [{month_str}] No VRT available -- will show placeholder")
        continue

    print(f"INFO  [{month_str}] Rendering PNG...")
    t0 = time.time()
    try:
        save_raw_panel_png(month_str, vrt_path, out_png, out_json, boundary_gdf)
        with open(out_json) as _f:
            _meta = json.load(_f)
        PANEL_PNGS[month_str]    = out_png
        PANEL_EXTENTS[month_str] = _meta["extent"]
        print(f"INFO  [{month_str}] Done in {time.time()-t0:.1f}s")
    except Exception as e:
        print(f"ERROR [{month_str}]: {e}")

print(f"\nINFO - Phase 1 complete: {len(PANEL_PNGS)}/12 panels ready")


## 8. Phase 2: Assemble 3x4 Publication Figure

Draws each panel with Cartopy (EPSG:4326), adds gridlines, boundary overlay,
and month labels. Missing months are shown as grey placeholders.


In [ ]:
# Determine global extent from all available panels
if PANEL_EXTENTS:
    all_xmin = min(e[0] for e in PANEL_EXTENTS.values())
    all_xmax = max(e[1] for e in PANEL_EXTENTS.values())
    all_ymin = min(e[2] for e in PANEL_EXTENTS.values())
    all_ymax = max(e[3] for e in PANEL_EXTENTS.values())
else:
    all_xmin, all_xmax, all_ymin, all_ymax = -180, 180, -90, 90

_proj = ccrs.PlateCarree()
fig, axes = plt.subplots(
    N_ROWS, N_COLS,
    figsize=FIGSIZE,
    subplot_kw={"projection": _proj},
    gridspec_kw={"hspace": 0.08, "wspace": 0.04},
)

for idx, month in enumerate(ALL_MONTHS):
    row, col = divmod(idx, N_COLS)
    ax = axes[row, col]
    month_str = f"{TARGET_YEAR}-{month}"
    label = f"{MONTH_ABBR[month]} {TARGET_YEAR}"

    ax.set_extent([all_xmin, all_xmax, all_ymin, all_ymax], crs=_proj)

    if month_str in PANEL_PNGS:
        xmin, xmax, ymin, ymax = PANEL_EXTENTS[month_str]
        img = plt.imread(str(PANEL_PNGS[month_str]))
        ax.imshow(img, origin="upper",
                  extent=[xmin, xmax, ymin, ymax],
                  transform=_proj, interpolation="bilinear")
    else:
        ax.set_facecolor("#404040")
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                ha="center", va="center", color="white",
                fontsize=BASE_FONT - 2)

    if boundary_gdf is not None:
        ax.add_geometries(
            boundary_gdf.geometry, crs=_proj,
            facecolor="none", edgecolor="white", linewidth=0.8,
        )

    ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="white", alpha=0.6)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, edgecolor="white", alpha=0.6)

    gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                      color="white", alpha=0.5, linestyle="--")
    gl.top_labels    = False
    gl.right_labels  = False
    gl.left_labels   = (col == 0)
    gl.bottom_labels = (row == N_ROWS - 1)
    gl.xformatter = LONGITUDE_FORMATTER
    gl.yformatter = LATITUDE_FORMATTER
    gl.xlabel_style = {"size": BASE_FONT - 4}
    gl.ylabel_style = {"size": BASE_FONT - 4}

    ax.set_title(label, fontsize=BASE_FONT, pad=4, fontweight="bold")

fig.text(
    0.5, 0.01,
    f"R={BAND_NAMES[0]}  G={BAND_NAMES[1]}  B={BAND_NAMES[2]}  "
    f"| Stretch: p=[{P_MIN},{P_MAX}]  gamma={GAMMA}  "
    f"| Direction: {DIRECTION}",
    ha="center", va="bottom", fontsize=BASE_FONT - 2, color="black",
)

_out_fig = Path(FIGURE_DIR) / f"S1-GRiTS_{Study_Area}_{TARGET_YEAR}_{DIRECTION}_3x4.png"
fig.savefig(str(_out_fig), dpi=DPI, bbox_inches="tight", facecolor="white")
plt.show()
print(f"INFO - Figure saved: {_out_fig}")
